# Turkish Morphology Retrieval — 5 Encoder Comparison

Bu notebook beş encoder'ı aynı train/dev bölünmesi, aynı simetrik çiftler ve aynı LoRA ayarlarıyla sırayla karşılaştırır. `RUN_SEALED_TEST=True  # Beş modelin tamamı development sonrası sealed 600'de raporlanır` olarak kalmalıdır; model seçimi yalnız development ile yapılır.

Pilot modunda mevcut pilot100 kullanılır; final modunda 1.250 family varsa 2.500 yönlü eğitim çifti oluşturulur. Bu, 2.500 bağımsız family değildir. Her model GPU belleği temizlenerek sırayla çalıştırılır.

In [ ]:
!pip uninstall -y -q torchvision torchao
!pip install -q -U sentence-transformers transformers peft accelerate datasets pandas
import gc, hashlib, json, os, random, shutil, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd, torch


In [ ]:
REPO_URL='https://github.com/TR-morph-retrieval/turkish-morph-retrieval.git'
ROOT=Path('/content/turkish-morph-retrieval')
if not (ROOT/'.git').exists(): subprocess.run(['git','clone','--depth','1',REPO_URL,str(ROOT)],check=True)
else: subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'],check=True)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
# Şimdilik mevcut pilotla çalışır; final üretimde PILOT_MODE=False yapıp accepted.jsonl kullanın.
PILOT_MODE=True
TRAIN_JSONL=ROOT/'train/data/pilot/pilot100.jsonl'
RUN_SEALED_TEST=True  # Beş modelin tamamı development sonrası sealed 600'de raporlanır
SEED=42
MODELS={
 'modernbert-tr': {'repo':'ytu-ce-cosmos/modernbert-tr-embed','q':'','d':'','targets':['Wqkv','Wo','Wi']},
 'bge-m3': {'repo':'BAAI/bge-m3','q':'','d':'','targets':['query','key','value','dense']},
 'e5-large': {'repo':'intfloat/multilingual-e5-large','q':'query: ','d':'passage: ','targets':['query','key','value','dense']},
 'trmteb-ft-110m': {'repo':'trmteb/turkish-embedding-model-fine-tuned','q':'','d':'','targets':['query','key','value','dense']},
 'multilingual-mpnet': {'repo':'sentence-transformers/paraphrase-multilingual-mpnet-base-v2','q':'','d':'','targets':['query','key','value','dense']}
}
PARAMS={'epochs':3,'batch_size':32,'cache_mini_batch':16,'learning_rate':2e-5,'warmup_ratio':.1,'weight_decay':.01,'lora_r':16,'lora_alpha':32,'lora_dropout':.05,'max_seq_length':256}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from datasets import Dataset
def stable_dev(fid): return int.from_bytes(hashlib.sha256(f'{SEED}:{fid}'.encode()).digest()[:4],'big')%10==0
def load_rows(path):
    if not path.exists(): raise FileNotFoundError(f'Final train JSONL bulunamadı: {path}')
    raw=[json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]
    if not raw: raise ValueError('Train JSONL boş')
    if PILOT_MODE:
        from train.production import validate_family
        before=len(raw); raw=[r for r in raw if not validate_family(r.get('family',{}))]
        print(f'Pilot filtresi: {len(raw)} kullanılacak, {before-len(raw)} eski/uyumsuz kayıt atlandı.')
    elif any(r.get('purpose')=='pilot_only' or r.get('eligible_for_final_train') is not True for r in raw):
        raise ValueError('Pilot/onaysız kayıt final karşılaştırmaya giremez')
    rows=[]
    for r in raw:
        f=r['family']; by={c['slot']:c['text'].strip() for c in f['candidates']}
        if set(by)!={'positive','morph_1','morph_2','semantic_1'}: raise ValueError('Aday şeması hatalı')
        fid=f.get('family_id') or r.get('pilot_review_id')
        rows.append({'family_id':fid,'target_feature':f['target_feature'],'query':f['query'].strip(),'positive':by['positive'],'morph_1':by['morph_1'],'morph_2':by['morph_2'],'semantic_1':by['semantic_1']})
    if len({r['family_id'] for r in rows})!=len(rows): raise ValueError('Tekrarlı family_id')
    return rows
ALL_ROWS=load_rows(TRAIN_JSONL)
DEV_ROWS=[r for r in ALL_ROWS if stable_dev(r['family_id'])]; TRAIN_ROWS=[r for r in ALL_ROWS if not stable_dev(r['family_id'])]
assert TRAIN_ROWS and DEV_ROWS
print('family:',len(ALL_ROWS),'train:',len(TRAIN_ROWS),'dev:',len(DEV_ROWS))
def samples_for(rows, cfg):
    q=lambda x:cfg['q']+x; d=lambda x:cfg['d']+x; out=[]
    for r in rows:
        out.append({'anchor':q(r['query']),'positive':d(r['positive']),'negative_1':d(r['morph_1']),'negative_2':d(r['morph_2']),'negative_3':d(r['semantic_1'])})
        out.append({'anchor':q(r['positive']),'positive':d(r['query']),'negative_1':d(r['morph_1']),'negative_2':d(r['morph_2']),'negative_3':d(r['semantic_1'])})
    return Dataset.from_list(out)


In [ ]:
from sentence_transformers import SentenceTransformer
def metrics(model, rows, cfg, batch_size=32):
    q=lambda x:cfg['q']+x; d=lambda x:cfg['d']+x
    qe=model.encode([q(r['query']) for r in rows],batch_size=batch_size,normalize_embeddings=True,convert_to_numpy=True)
    docs=[d(r[k]) for r in rows for k in ('positive','morph_1','morph_2','semantic_1')]
    de=model.encode(docs,batch_size=batch_size,normalize_embeddings=True,convert_to_numpy=True).reshape(len(rows),4,-1)
    scores=np.einsum('bd,bkd->bk',qe,de); ranks=np.array([1+np.sum(s[1:]>s[0]) for s in scores])
    return {'families':len(rows),'recall@1':float(np.mean(ranks<=1)),'recall@3':float(np.mean(ranks<=3)),'mrr@10':float(np.mean(1/ranks)),'mean_rank':float(np.mean(ranks)),'morph_pairwise':float(np.mean(scores[:,0,None]>scores[:,1:3]))}
def targets_for(model, wanted):
    names={name.rsplit('.',1)[-1] for name,_ in model._first_module().auto_model.named_modules()}
    found=[x for x in wanted if x in names]
    if not found: raise RuntimeError(f'LoRA hedefi bulunamadı: {wanted}; örnek modüller={sorted(names)[:30]}')
    return found


In [ ]:
from peft import LoraConfig, get_peft_model
from sentence_transformers.sentence_transformer.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
def train_one(key,cfg):
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model=SentenceTransformer(cfg['repo'],trust_remote_code=True,model_kwargs={'torch_dtype':dtype}); model.max_seq_length=PARAMS['max_seq_length']
    base=metrics(model,DEV_ROWS,cfg)
    first=model._first_module(); targets=targets_for(model,cfg['targets'])
    first.auto_model=get_peft_model(first.auto_model,LoraConfig(r=PARAMS['lora_r'],lora_alpha=PARAMS['lora_alpha'],lora_dropout=PARAMS['lora_dropout'],bias='none',target_modules=targets,task_type='FEATURE_EXTRACTION'))
    loss=CachedMultipleNegativesRankingLoss(model,scale=20.0,mini_batch_size=PARAMS['cache_mini_batch'])
    args=SentenceTransformerTrainingArguments(output_dir=f'/content/compare_{key}',num_train_epochs=PARAMS['epochs'],per_device_train_batch_size=PARAMS['batch_size'],learning_rate=PARAMS['learning_rate'],warmup_ratio=PARAMS['warmup_ratio'],weight_decay=PARAMS['weight_decay'],fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),batch_sampler=BatchSamplers.NO_DUPLICATES,logging_steps=10,save_strategy='no',seed=SEED,data_seed=SEED,report_to='none')
    trainer=SentenceTransformerTrainer(model=model,args=args,train_dataset=samples_for(TRAIN_ROWS,cfg),loss=loss); trainer.train()
    tuned=metrics(model,DEV_ROWS,cfg)
    sealed=None
    if RUN_SEALED_TEST:
        from test.evaluation import load_items, score_encoder
        sealed_path=ROOT/'test/data/morph_test_600_sealed.json'
        sealed_items=load_items(sealed_path)
        assert len(sealed_items)==600 and all(x.get('target_split')=='sealed_test' for x in sealed_items)
        sealed=score_encoder(model,sealed_items,query_prefix=cfg['q'],document_prefix=cfg['d'],batch_size=32)['summary']
    out={'model':key,'repo':cfg['repo'],'zero_shot':base,'fine_tuned':tuned,'sealed_test':sealed}
    del trainer,loss,model; gc.collect(); torch.cuda.empty_cache(); return out
results=[]
for key,cfg in MODELS.items():
    print('\n===',key,'===')
    try: results.append(train_one(key,cfg))
    except Exception as exc: results.append({'model':key,'error':repr(exc)}); print('SKIP:',exc)
comparison=[]
for r in results:
    if 'error' in r: comparison.append({'model':r['model'],'error':r['error']}); continue
    row={'model':r['model']}
    for k,v in r['zero_shot'].items(): row['zero_'+k]=v
    for k,v in r['fine_tuned'].items(): row['tuned_'+k]=v
    if r.get('sealed_test'):
        for k,v in r['sealed_test'].items():
            if isinstance(v,(int,float)): row['sealed_'+k]=v
    comparison.append(row)
comparison=pd.DataFrame(comparison).sort_values('tuned_mrr@10',ascending=False); display(comparison)
Path('/content/encoder_comparison_results.json').write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding='utf-8')

## Sealed test protokolü

Beş model de aynı train/dev protokolüyle eğitildikten sonra sealed 600'de raporlanır. Buradaki amaç model seçip testi tekrar tekrar denemek değil, önceden belirlenmiş beş encoder ailesini tek seferde karşılaştırmaktır. En iyi modelin ek hiperparametre ayarı sealed sonuçlarına bakılarak yapılmamalıdır.
